# Nickel purification with optional SEC

Nickel Phytip purification on the Hamilton STAR, followed by an optional DPX size-exclusion cleanup. SEC aspirates from the nickel elution plate, so there is no separate SEC sample plate.

Protein Select is not in this method. Elution is at room temperature: the Opentrons temperature module is left out until pylabrobot supports it again. The nickel elution plate sits on an MFX deep-well module in the back site of the nickel carrier.

Pierce every SEC tip that will be used before Setup Deck. The 96-head mounts the whole SEC rack for the column push, so leave unused SEC positions empty.

**Workflow:** set the sample mode and, if you want the cleanup, check SEC. Click Setup Deck, teach the SEC load height on an empty tip, return that tip, then Run purification. Changing the sample count, rails, or the SEC checkbox changes which labware is created. Restart the kernel before Setup Deck if you change those.

Slot 1 is the front of a carrier and slot 5 is the back. Default rails match the previous nickel deck, with the old Select plate rail used for the SEC plates.

| Carrier | Default rail | Slots, front to back |
| --- | --- | --- |
| Nickel plates, MFX deep-well modules | 1 | Equilibration, sample, wash 1, wash 2, elution |
| Tips, `TIP_CAR_480_A00` | 7 | IMAC Phytips, SEC tips, 1000 µL buffer tips, 300 µL SEC transfer tips, optional second 300 µL rack |
| Nickel regeneration, `PLT_CAR_L5AC_A00` | 13 | Strip, rinse, recharge, rinse, storage |
| SEC plates, `PLT_CAR_L5AC_A00` | 19 | Swelling buffer, waste, SEC elution. Present only when SEC cleanup is checked |
| Sample tubes | 35, 37, 39, 41 | 50 mL tubes, up to 12 per carrier |
| Buffer troughs | 43 | Nickel wash, nickel elution |

SEC sample volume is taken from the nickel elution and has to be within the 300 µL transfer tips and no larger than `VOL_ELN`. Buffer transfer still adds 200 µL extra elution buffer to that plate before the Phytip elution mix.


In [ ]:
import asyncio
import io
import sys
import time
import traceback
from contextlib import redirect_stdout

import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, r"C:\Users\16122\Documents\Github\pylabrobot")
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.liquid_handling.liquid_classes.hamilton import HamiltonLiquidClass
from pylabrobot.resources import (
    Coordinate,
    Cor_96_wellplate_2mL_Vb,
    Cor_Axy_24_wellplate_10mL_Vb,
    Cor_Falcon_tube_50mL_Vb,
    MFX_DWP_module_flat,
    PLT_CAR_L5AC_A00,
    Resource,
    TIP_CAR_480_A00,
    Trough_CAR_5R60_A00,
    Well,
    hamilton_1_trough_60ml_Vb,
    hamilton_96_tiprack_1000uL,
    hamilton_96_tiprack_1000uL_filter_ultrawide,
    hamilton_96_tiprack_1000uL_filter_wide,
    hamilton_96_tiprack_300uL,
    hamilton_mfx_carrier_L5_base,
    hamilton_tube_carrier_12_b00,
    nest_1_troughplate_195000uL_Vb,
    set_tip_tracking,
    set_volume_tracking,
)
from pylabrobot.resources.hamilton import STARDeck

deck_is_ready = False
lh = None
LOAD_WELLS = {}
teach_tip_spot = None
status_output = None


In [ ]:
# DPX Table 1. "Void" is the top of each bin, which matches the catalog sample volume.
# The procedure's worked example (250 uL column, 5,000 Da, 688 uL swell) is the last row.
SEC_COLUMNS = {
    "1500 Da, void 100-150 uL (swell 263)": {"void_uL": 150, "swell_uL": 263},
    "1500 Da, void 150-200 uL (swell 394)": {"void_uL": 200, "swell_uL": 394},
    "1500 Da, void 200-250 uL (swell 525)": {"void_uL": 250, "swell_uL": 525},
    "1500 Da, void 250-300 uL (swell 656)": {"void_uL": 300, "swell_uL": 656},
    "5000 Da, void 100-150 uL (swell 275)": {"void_uL": 150, "swell_uL": 275},
    "5000 Da, void 150-200 uL (swell 408)": {"void_uL": 200, "swell_uL": 408},
    "5000 Da, void 200-250 uL (swell 550)": {"void_uL": 250, "swell_uL": 550},
    "5000 Da, void 250-300 uL (swell 688)": {"void_uL": 300, "swell_uL": 688},
}
DEFAULT_COLUMN = "5000 Da, void 250-300 uL (swell 688)"

number_of_samples = 8
samples_in_50ml = False
samples_in_24well = False
samples_in_24well_offset = False
wash_in_24well_offset = False
sample_volume = 5  # mL, recorded on the run summary
first_well = "A1"
RUN_SEC = False

tip_carrier_rail = 7
nickel_plate_carrier_rail = 1
nickel_regen_carrier_rail = 13
sec_plate_carrier_rail = 19
tube_carrier_rail_1 = 35
tube_carrier_rail_2 = 37
tube_carrier_rail_3 = 39
tube_carrier_rail_4 = 41
trough_carrier_rail = 43

# Tip-carrier slots are 1 at the front through 5 at the back.
imac_tip_pos = 1
sec_tip_pos = 2
buffer_transfer_tip_pos = 3
sec_transfer_tip_pos = 4
sec_transfer_tip_pos_2 = 0  # 0 leaves the second 300 uL rack off the deck
sec_buffer_plate_pos = 1
sec_waste_plate_pos = 2
sec_elution_plate_pos = 3

# Nickel volumes in uL. Buffer transfer adds 200 uL on top of these.
VOL_EQL = 800
VOL_SAMPLE = 800
VOL_WASH = 800
VOL_ELN = 400

CYCLES_EQL = 2
CYCLES_SAMPLE = 5
CYCLES_WASH = 3
CYCLES_ELN = 3
NICKEL_REPEATS = 1

# uL/s. 4 uL/s is 240 uL/min.
FLOWRATE = 9
DELAY = 9.9

# SEC volumes and timing. Sample is aspirated from the nickel elution plate.
swell_uL = 688
void_uL = 300
sample_uL = 250
elution_uL = 250
correction_uL = 0
do_void_elution = True
do_buffer_touchoff = True

column_flow = 5.0
transfer_flow = 75.0
aspirate_flow = 100.0
swell_hold_s = 150.0
flow_hold_s = 30.0
drop_hold_s = 10.0
touch_hold_s = 5.0
drop_uL = 25.0
touch_air_uL = 20.0
airgap_margin_uL = 50.0

load_depth_mm = 6.0
buffer_height_mm = 5.0
sample_height_mm = 2.0
flow_height_mm = 15.0

DEVELOPMENT_MODE = False
FIRMWARE_MAX_SETTLE_S = 9.9


In [ ]:
def log(message: str) -> None:
    text = message if message.endswith("\n") else message + "\n"
    if status_output is not None:
        status_output.append_stdout(text)
    else:
        print(text, end="")


def plate_wells(plate_name: str, names: list[str]):
    plate = lh.deck.get_resource(plate_name)
    return [plate[name][0] for name in names]


def tip_spots(rack_name: str, names: list[str]):
    rack = lh.deck.get_resource(rack_name)
    return [rack[name][0] for name in names]


def sample_names(n: int, start: str) -> list[str]:
    """Column-wise 96-well names. Later columns start again at row A."""
    rows = list("ABCDEFGH")
    start = start.strip().upper()
    row_i = rows.index(start[0])
    col = int(start[1:])
    names = []
    for column in range(col, 13):
        first_row = row_i if column == col else 0
        for row in rows[first_row:]:
            names.append(f"{row}{column}")
            if len(names) == n:
                return names
    raise ValueError(f"Only {len(names)} wells are left from {start}.")


def fifty_ml_wells() -> list[str]:
    """Tip positions loaded from 50 mL tubes.

    Every column starts at the same row as the first well. That is the order the
    nickel 8-channel loader uses.
    """
    rows = list("ABCDEFGH")
    rows = rows[rows.index(first_well[0]):]
    first_column = int(first_well[1:])
    wells = [f"{row}{col}" for col in range(first_column, 13) for row in rows]
    return wells[:number_of_samples]


def groups_24well() -> list[dict]:
    """8-channel groups for a 24-well sample plate.

    Plate column 1 uses tips A-D of a 96-rack column, and the next plate column
    uses tips E-H of that same rack column.
    """
    rows = list("ABCD")
    rows = rows[rows.index(first_well[0]):]
    remaining = number_of_samples
    col = int(first_well[1:])
    first_column = col
    groups = []
    while remaining > 0 and col <= 6:
        plate_wells_in_col = [f"{row}{col}" for row in rows][:remaining]
        rel = col - first_column
        tip_col = 1 + rel // 2
        if rel % 2 == 0:
            tip_start, tip_end = "A", "D"
            tip_rows = list("ABCD")
        else:
            tip_start, tip_end = "E", "H"
            tip_rows = list("EFGH")
        n = len(plate_wells_in_col)
        groups.append(
            {
                "span": f"{tip_start}{tip_col}:{tip_end}{tip_col}",
                "plate_wells": plate_wells_in_col,
                "elution_wells": [f"{row}{tip_col}" for row in tip_rows[:n]],
            }
        )
        remaining -= n
        col += 1
    return groups


def elution_wells() -> list[str]:
    """Wells on the nickel elution plate that hold the loaded samples."""
    if samples_in_24well:
        names = [well for group in groups_24well() for well in group["elution_wells"]]
    elif samples_in_50ml:
        names = fifty_ml_wells()
    else:
        names = sample_names(number_of_samples, first_well)
    if len(names) != number_of_samples:
        raise ValueError(
            f"Could only map {len(names)} of {number_of_samples} samples from {first_well}."
        )
    return names


def column_groups(names: list[str]) -> list[list[str]]:
    """Split well names into same-column groups so the 8 channels share one X."""
    groups: list[list[str]] = []
    current: list[str] = []
    for name in names:
        if current and name[1:] != current[0][1:]:
            groups.append(current)
            current = []
        current.append(name)
    if current:
        groups.append(current)
    return groups


def transfer_tip_columns_needed(names: list[str]) -> int:
    """Sample transfer and elution each take one tip column per plate column."""
    return 2 * len(column_groups(names))


# Arm travel added on top of plunger time and programmed waits.
_HEAD_TRAVEL_S = 8.0
_CHANNEL_TRAVEL_S = 8.0
_PICKUP96_S = 12.0
_PICKUP8_S = 8.0
# Flow used when buffer transfer does not pass a flow rate. These are the STAR 96-head defaults.
_BUFFER_ASPIRATE_FLOW = 250.0
_BUFFER_DISPENSE_FLOW = 120.0


def format_duration(seconds: float) -> str:
    total = max(0, int(round(seconds)))
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    if hours and minutes:
        return f"{hours} h {minutes} min"
    if hours:
        return f"{hours} h"
    if minutes and secs and minutes < 10:
        return f"{minutes} min {secs} s"
    if minutes:
        return f"{minutes} min"
    return f"{secs} s"


def _mix96_seconds(cycles: int, volume: float, flow: float, delay: float) -> float:
    flow = max(float(flow), 0.1)
    cycle = volume / flow + delay + _HEAD_TRAVEL_S + volume / flow + _HEAD_TRAVEL_S
    return cycles * cycle


def _channel_mix_seconds(cycles: int, volume: float, flow: float, delay: float) -> float:
    flow = max(float(flow), 0.1)
    cycle = volume / flow + delay + _CHANNEL_TRAVEL_S + volume / flow + _CHANNEL_TRAVEL_S
    return cycles * cycle


def _estimate_elution_names(settings: dict) -> list[str]:
    n = int(settings["number_of_samples"])
    start = str(settings["first_well"]).strip().upper()
    if settings["samples_in_24well"]:
        rows = list("ABCD")[list("ABCD").index(start[0]):]
        remaining = n
        col = int(start[1:])
        first_column = col
        names = []
        while remaining > 0 and col <= 6:
            take = min(len(rows), remaining)
            rel = col - first_column
            tip_col = 1 + rel // 2
            tip_rows = list("ABCD") if rel % 2 == 0 else list("EFGH")
            names.extend(f"{row}{tip_col}" for row in tip_rows[:take])
            remaining -= take
            col += 1
        return names
    if settings["samples_in_50ml"]:
        rows = list("ABCDEFGH")[list("ABCDEFGH").index(start[0]):]
        first_column = int(start[1:])
        wells = [f"{row}{col}" for col in range(first_column, 13) for row in rows]
        return wells[:n]
    return sample_names(n, start)


def estimate_run_seconds(settings: dict, transfer_buffers: bool, regenerate: bool) -> float:
    """Seconds for liquid flow, programmed waits, and a fixed travel allowance per move."""
    flow = float(settings["flowrate"])
    delay = float(settings["delay"])
    repeats = int(settings["nickel_repeats"])
    channel_load = bool(settings["samples_in_50ml"] or settings["samples_in_24well"])
    total = 0.0

    if transfer_buffers:
        volumes = [
            int(settings["vol_eql"]) + 200,
            int(settings["vol_wash"]) + 200,
            int(settings["vol_wash"]) + 200,
            int(settings["vol_eln"]) + 200,
        ]
        total += _PICKUP96_S + _PICKUP96_S
        for volume in volumes:
            total += volume / _BUFFER_ASPIRATE_FLOW + _HEAD_TRAVEL_S
            total += volume / _BUFFER_DISPENSE_FLOW + _HEAD_TRAVEL_S

    sample_groups = 1
    if settings["samples_in_50ml"]:
        sample_groups = -(-int(settings["number_of_samples"]) // 8)
    elif settings["samples_in_24well"]:
        per_column = len(list("ABCD")[list("ABCD").index(str(settings["first_well"]).strip().upper()[0]):])
        sample_groups = -(-int(settings["number_of_samples"]) // per_column)

    nickel = _PICKUP96_S + _PICKUP96_S + _HEAD_TRAVEL_S
    nickel += _mix96_seconds(int(settings["cycles_eql"]), int(settings["vol_eql"]), flow, delay)
    if channel_load:
        nickel += _PICKUP96_S + _PICKUP96_S
        nickel += sample_groups * (
            _PICKUP8_S
            + _PICKUP8_S
            + _channel_mix_seconds(int(settings["cycles_sample"]), int(settings["vol_sample"]), flow, delay)
        )
    else:
        nickel += _mix96_seconds(int(settings["cycles_sample"]), int(settings["vol_sample"]), flow, delay)
    nickel += _mix96_seconds(int(settings["cycles_wash"]), int(settings["vol_wash"]), flow, delay)
    nickel += _mix96_seconds(int(settings["cycles_wash"]), int(settings["vol_wash"]), flow, delay)
    nickel += _mix96_seconds(int(settings["cycles_eln"]), int(settings["vol_eln"]), flow, delay)
    total += repeats * nickel

    if settings["run_sec"]:
        names = _estimate_elution_names(settings)
        columns = max(1, len(column_groups(names)))
        column_flow = max(float(settings["column_flow"]), 0.1)
        aspirate_flow = max(float(settings["aspirate_flow"]), 0.1)
        transfer_flow = max(float(settings["transfer_flow"]), 0.1)
        swell = float(settings["swell_uL"])
        sample = float(settings["sample_uL"])
        elution = float(settings["elution_uL"])
        void_liquid = (
            max(0.0, float(settings["void_uL"]) - sample) if settings["do_void_elution"] else 0.0
        )

        def transfer_s(volume: float) -> float:
            return (
                _PICKUP8_S
                + volume / aspirate_flow
                + _CHANNEL_TRAVEL_S
                + volume / transfer_flow
                + _CHANNEL_TRAVEL_S
                + _PICKUP8_S
            )

        def push_s(volume: float) -> float:
            touch = float(settings["touch_hold_s"]) if float(settings["touch_air_uL"]) > 0 else 0.0
            return _PICKUP96_S + _HEAD_TRAVEL_S + volume / column_flow + float(settings["flow_hold_s"]) + touch + _PICKUP96_S

        for _column in range(columns):
            total += _PICKUP8_S + swell / column_flow + _CHANNEL_TRAVEL_S + float(settings["swell_hold_s"])
            if settings["do_buffer_touchoff"]:
                total += _CHANNEL_TRAVEL_S
            total += _PICKUP8_S
            total += transfer_s(sample)
        total += push_s(sample + float(settings["correction_uL"]))
        if void_liquid > 0:
            total += columns * transfer_s(void_liquid)
            total += push_s(void_liquid + float(settings["correction_uL"]))
        total += columns * transfer_s(elution)
        total += push_s(elution + float(settings["correction_uL"]))

    if regenerate:
        total += _PICKUP96_S + _PICKUP96_S
        total += _mix96_seconds(2, 800, flow, 1)
        total += _mix96_seconds(4, 800, flow, 5)
        total += _mix96_seconds(4, 800, flow, 5)
        total += _mix96_seconds(4, 800, flow, 5)
        total += 200 / 1.0 + _HEAD_TRAVEL_S

    return total


def repeat(value, n: int):
    return [value] * n


def end_z(resource, liquid_height: float = 0.0, z_offset: float = 0.0) -> float:
    """Absolute Z of the tip end, using the same origin as STAR aspirate/dispense."""
    return resource.get_location_wrt(lh.deck).z + resource.material_z_thickness + z_offset + liquid_height


def unique_positions(kind: str, positions: list[int]) -> None:
    if any(position < 1 or position > 5 for position in positions):
        raise ValueError(f"{kind} positions must be 1 (front) through 5 (back).")
    if len(set(positions)) != len(positions):
        raise ValueError(f"{kind} positions overlap: {positions}.")


def place_on_carrier(carrier, placements, kind: str) -> None:
    """Place resources on carrier slots numbered 1 (front) through 5 (back)."""
    positions = [position for position, _resource in placements]
    unique_positions(kind, positions)
    for position, resource in placements:
        carrier[position - 1] = resource


def zero_air_liquid_class(dispense_transport_air: float) -> HamiltonLiquidClass:
    """Identity correction and no extra air, so the programmed microlitres are the plunger move."""
    return HamiltonLiquidClass(
        curve={0.0: 0.0, 1000.0: 1000.0},
        aspiration_flow_rate=column_flow,
        aspiration_mix_flow_rate=column_flow,
        aspiration_air_transport_volume=0.0,
        aspiration_blow_out_volume=0.0,
        aspiration_swap_speed=10.0,
        aspiration_settling_time=0.0,
        aspiration_over_aspirate_volume=0.0,
        aspiration_clot_retract_height=0.0,
        dispense_flow_rate=column_flow,
        dispense_mode=2.0,
        dispense_mix_flow_rate=column_flow,
        dispense_air_transport_volume=dispense_transport_air,
        dispense_blow_out_volume=0.0,
        dispense_swap_speed=10.0,
        dispense_settling_time=0.0,
        dispense_stop_flow_rate=5.0,
        dispense_stop_back_volume=0.0,
    )


def sec_opening_above_spot(tip) -> float:
    """Height of the SEC tip opening above the tip-spot origin.

    The tip rests by its collar on the spot. The opening is one fitting depth below the top.
    """
    if tip.location is not None:
        return tip.location.z + tip.get_size_z() - tip.fitting_depth
    collar = tip.collar_height if getattr(tip, "has_collar_height", False) else 0.0
    pick_up = tip.pick_up_location or tip.get_anchor("c", "c", "t")
    return collar - pick_up.z + tip.get_size_z() - tip.fitting_depth


def build_load_guide(sec_rack):
    """Wells whose bottoms sit at the SEC tip opening, centred on each tip spot.

    Dispense into these wells with a negative Z offset to land just below the pierced barrier.
    """
    prototype = sec_rack["A1"][0].get_tip()
    opening_above_spot = sec_opening_above_spot(prototype)
    guide = Resource(
        name="sec_load_guide",
        size_x=122.4,
        size_y=82.6,
        size_z=20.0,
        category="plate",
    )
    wells = {}
    well_size = 9.0
    for row in "ABCDEFGH":
        for col in range(1, 13):
            spot = sec_rack[f"{row}{col}"][0]
            well = Well(
                name=f"sec_load_{row}{col}",
                size_x=well_size,
                size_y=well_size,
                size_z=1.0,
                material_z_thickness=0.0,
                max_volume=1000,
            )
            guide.assign_child_resource(
                well,
                location=Coordinate(
                    spot.location.x + spot.get_size_x() / 2 - well_size / 2,
                    spot.location.y + spot.get_size_y() / 2 - well_size / 2,
                    spot.location.z + opening_above_spot,
                ),
            )
            wells[f"{row}{col}"] = well
    return guide, wells, prototype, opening_above_spot


def build_deck_resources():
    """Create the carriers for this run. SEC racks are included only when SEC cleanup is on."""
    tip_car = TIP_CAR_480_A00(name="tip carrier")
    tip_placements = [
        (imac_tip_pos, hamilton_96_tiprack_1000uL_filter_wide(name="imac_tips")),
        (buffer_transfer_tip_pos, hamilton_96_tiprack_1000uL(name="buffer_transfer_tips")),
    ]
    if RUN_SEC:
        tip_placements.append((sec_tip_pos, hamilton_96_tiprack_1000uL_filter_ultrawide(name="sec_tips")))
        tip_placements.append((sec_transfer_tip_pos, hamilton_96_tiprack_300uL(name="sec_transfer_tips")))
        if sec_transfer_tip_pos_2:
            tip_placements.append(
                (sec_transfer_tip_pos_2, hamilton_96_tiprack_300uL(name="sec_transfer_tips_2"))
            )
    place_on_carrier(tip_car, tip_placements, "Tip")

    nickel_plates = hamilton_mfx_carrier_L5_base(
        name="nickel_plate_carrier",
        modules={i: MFX_DWP_module_flat(name=f"plate_module_{i + 1}") for i in range(5)},
    )
    nickel_plates[0] = Cor_96_wellplate_2mL_Vb(name="nickel_equilibration_plate")
    if samples_in_24well:
        nickel_plates[1] = Cor_Axy_24_wellplate_10mL_Vb(name="nickel_sample_plate")
    else:
        nickel_plates[1] = Cor_96_wellplate_2mL_Vb(name="nickel_sample_plate")
    nickel_plates[2] = Cor_96_wellplate_2mL_Vb(name="nickel_wash1_plate")
    nickel_plates[3] = Cor_96_wellplate_2mL_Vb(name="nickel_wash2_plate")
    nickel_plates[4] = Cor_96_wellplate_2mL_Vb(name="nickel_elution_plate")

    nickel_regen = PLT_CAR_L5AC_A00(name="nickel_regen_plate_carrier")
    nickel_regen[0] = nest_1_troughplate_195000uL_Vb(name="nickel_strip_plate")
    nickel_regen[1] = nest_1_troughplate_195000uL_Vb(name="nickel_rinse_plate")
    nickel_regen[2] = nest_1_troughplate_195000uL_Vb(name="nickel_recharge_plate")
    nickel_regen[3] = nest_1_troughplate_195000uL_Vb(name="rinse_plate")
    nickel_regen[4] = nest_1_troughplate_195000uL_Vb(name="storage_buffer_plate")

    sec_plates = None
    if RUN_SEC:
        sec_plates = PLT_CAR_L5AC_A00(name="sec_plate_carrier")
        place_on_carrier(
            sec_plates,
            [
                (sec_buffer_plate_pos, Cor_96_wellplate_2mL_Vb(name="sec_buffer_plate")),
                (sec_waste_plate_pos, Cor_96_wellplate_2mL_Vb(name="sec_waste_plate")),
                (sec_elution_plate_pos, Cor_96_wellplate_2mL_Vb(name="sec_elution_plate")),
            ],
            "SEC plate",
        )

    tube_carriers = []
    tube_plan = [
        (0, tube_carrier_rail_1, "tube_carrier1"),
        (12, tube_carrier_rail_2, "tube_carrier2"),
        (24, tube_carrier_rail_3, "tube_carrier3"),
        (36, tube_carrier_rail_4, "tube_carrier4"),
    ]
    for start, rail, name in tube_plan:
        if number_of_samples <= start:
            break
        carrier = hamilton_tube_carrier_12_b00(name=name)
        last = min(number_of_samples, start + 12)
        for i, tube_num in zip(range(11, 11 - (last - start), -1), range(start, last)):
            carrier[i] = Cor_Falcon_tube_50mL_Vb(name=f"sample_{tube_num}")
        tube_carriers.append((rail, carrier))

    trough_car = Trough_CAR_5R60_A00(name="trough_carrier")
    for i, stock_name in enumerate(("nickel_wash_buffer", "nickel_elute_buffer")):
        trough_car[i] = hamilton_1_trough_60ml_Vb(name=stock_name)

    return {
        "tip_carrier": tip_car,
        "nickel_plates": nickel_plates,
        "nickel_regen": nickel_regen,
        "sec_plates": sec_plates,
        "tube_carriers": tube_carriers,
        "trough_carrier": trough_car,
    }


async def setup_deck():
    global lh, deck_is_ready, LOAD_WELLS

    if deck_is_ready or lh is not None:
        return "Deck setup already started. Restart the kernel to set it up again."

    lh = LiquidHandler(
        backend=STARBackend(read_timeout=2400),
        deck=STARDeck(core_grippers="1000uL-5mL-on-waste"),
    )
    await lh.setup(skip_iswap=True)
    # Volume tracking stays off: the SEC dispense pushes air, not a tracked liquid.
    # Tip tracking stays off so parked Phytips and SEC tips can be picked up again.
    set_volume_tracking(enabled=False)
    set_tip_tracking(enabled=False)

    lh.deck.get_resource("trash_core96").location = Coordinate(-260, 106, 216.4)

    resources = build_deck_resources()
    lh.deck.assign_child_resource(resources["tip_carrier"], rails=tip_carrier_rail)
    lh.deck.assign_child_resource(resources["nickel_plates"], rails=nickel_plate_carrier_rail)
    lh.deck.assign_child_resource(resources["nickel_regen"], rails=nickel_regen_carrier_rail)
    for rail, carrier in resources["tube_carriers"]:
        lh.deck.assign_child_resource(carrier, rails=rail)
    lh.deck.assign_child_resource(resources["trough_carrier"], rails=trough_carrier_rail)

    note = "Nickel elution plate is on MFX deep-well module 5 (back of the nickel carrier)."
    if RUN_SEC:
        lh.deck.assign_child_resource(resources["sec_plates"], rails=sec_plate_carrier_rail)
        sec_rack = lh.deck.get_resource("sec_tips")
        guide, LOAD_WELLS, prototype, opening_above_spot = build_load_guide(sec_rack)
        # Parent the guide to the tip rack. A deck-level copy sits inside the carrier
        # footprint and STARDeck rejects that as a collision.
        sec_rack.assign_child_resource(guide, location=Coordinate.zero())
        note += (
            f" SEC tip opening is {opening_above_spot:.2f} mm above the tip-spot origin "
            f"(collar {prototype.collar_height:.2f} mm, length {prototype.get_size_z():.2f} mm, "
            f"fitting depth {prototype.fitting_depth:.2f} mm). "
            "Load depth is measured down from that opening."
        )

    deck_is_ready = True
    buf = io.StringIO()
    with redirect_stdout(buf):
        lh.summary()
    summary = buf.getvalue().strip()
    return {"status": "Deck setup complete.\n\n" + note, "deck": summary}


In [ ]:
async def plate_mix(cycles, volume, plate_loc, delay_pipette, flowrate, plate_offset=Coordinate(0, 0, 0)):
    for _ in range(cycles):
        await lh.aspirate96(
            lh.deck.get_resource(plate_loc),
            volume=volume,
            flow_rate=flowrate,
            settling_time=delay_pipette,
            offset=Coordinate(0, 0, -6) + plate_offset,
        )
        await lh.dispense96(
            lh.deck.get_resource(plate_loc),
            volume=volume,
            flow_rate=flowrate,
            offset=Coordinate(0, 0, 15) + plate_offset,
        )


async def sample_mix_50ml(cycles, volume, tubes, delay_pipette, flowrate):
    # Aspirate inside the 50 mL tube and dispense near the top.
    for _ in range(cycles):
        await lh.aspirate(
            tubes,
            vols=[volume] * len(tubes),
            flow_rates=[flowrate] * len(tubes),
            settling_time=[delay_pipette] * len(tubes),
            offsets=[Coordinate(0, 0, 20)] * len(tubes),
        )
        await lh.dispense(
            tubes,
            vols=[volume] * len(tubes),
            flow_rates=[flowrate] * len(tubes),
            offsets=[Coordinate(0, 0, 80)] * len(tubes),
        )


async def sample_mix_24well(cycles, volume, wells, delay_pipette, flowrate):
    for _ in range(cycles):
        await lh.aspirate(
            wells,
            vols=[volume] * len(wells),
            flow_rates=[flowrate] * len(wells),
            settling_time=[delay_pipette] * len(wells),
        )
        await lh.dispense(
            wells,
            vols=[volume] * len(wells),
            flow_rates=[flowrate] * len(wells),
            offsets=[Coordinate(0, 0, 15)] * len(wells),
        )


async def transfer_buffers_step(enabled: bool):
    if not enabled:
        return

    await lh.pick_up_tips96(lh.deck.get_resource("buffer_transfer_tips"))
    await lh.aspirate96(lh.deck.get_resource("nickel_wash_buffer"), volume=VOL_EQL + 200)
    await lh.dispense96(lh.deck.get_resource("nickel_equilibration_plate"), volume=VOL_EQL + 200)
    await lh.aspirate96(lh.deck.get_resource("nickel_wash_buffer"), volume=VOL_WASH + 200)
    await lh.dispense96(lh.deck.get_resource("nickel_wash1_plate"), volume=VOL_WASH + 200)
    await lh.aspirate96(lh.deck.get_resource("nickel_wash_buffer"), volume=VOL_WASH + 200)
    await lh.dispense96(lh.deck.get_resource("nickel_wash2_plate"), volume=VOL_WASH + 200)
    await lh.aspirate96(lh.deck.get_resource("nickel_elute_buffer"), volume=VOL_ELN + 200)
    await lh.dispense96(lh.deck.get_resource("nickel_elution_plate"), volume=VOL_ELN + 200)
    await lh.drop_tips96(lh.deck.get_resource("buffer_transfer_tips"))


async def regenerate_columns_step(enabled: bool):
    if not enabled:
        return

    await lh.pick_up_tips96(lh.deck.get_resource("imac_tips"))
    log("Stripping nickel columns in GuHCl")
    await plate_mix(2, 800, "nickel_strip_plate", 1, FLOWRATE)
    log("Washing nickel columns")
    await plate_mix(4, 800, "nickel_rinse_plate", 5, FLOWRATE)
    log("Recharging nickel columns")
    await plate_mix(4, 800, "nickel_recharge_plate", 5, FLOWRATE)
    log("Washing nickel columns")
    await plate_mix(4, 800, "rinse_plate", 5, FLOWRATE)
    log("Storing nickel columns")
    await lh.aspirate96(
        lh.deck.get_resource("storage_buffer_plate"),
        volume=200,
        flow_rate=1,
        offset=Coordinate(0, 0, -8),
    )
    await lh.drop_tips96(lh.deck.get_resource("imac_tips"), allow_nonzero_volume=True)


async def run_nickel():
    tip_set = []
    tube_set = []

    if samples_in_50ml:
        wells = fifty_ml_wells()
        if len(wells) != number_of_samples:
            raise ValueError(f"With first well {first_well}, 50 mL loading can reach {len(wells)} samples.")
        tube_list = [lh.deck.get_resource(f"sample_{i}") for i in range(number_of_samples)]
        for i in range(0, len(tube_list), 8):
            chunk_tube = tube_list[i : i + 8]
            tube_set.append(chunk_tube)
            chunk_tip = wells[i : i + len(chunk_tube)]
            tip_set.append(chunk_tip[0] + ":" + chunk_tip[-1])

    elif samples_in_24well:
        groups = groups_24well()
        loaded = [well for group in groups for well in group["plate_wells"]]
        if len(loaded) != number_of_samples:
            raise ValueError(f"With first well {first_well}, 24-well loading can reach {len(loaded)} samples.")
        plate = lh.deck.get_resource("nickel_sample_plate")
        for group in groups:
            tube_set.append([plate[well][0] for well in group["plate_wells"]])
            tip_set.append(group["span"])

    plate_offset = Coordinate(5, -5, 0) if wash_in_24well_offset else Coordinate(0, 0, 0)
    for _ in range(NICKEL_REPEATS):
        await lh.pick_up_tips96(lh.deck.get_resource("imac_tips"))
        log("Equilibrating nickel columns")
        await plate_mix(CYCLES_EQL, VOL_EQL, "nickel_equilibration_plate", DELAY, FLOWRATE, plate_offset=plate_offset)
        if samples_in_50ml or samples_in_24well:
            await lh.drop_tips96(lh.deck.get_resource("imac_tips"))
            for tips, tubes in zip(tip_set, tube_set):
                await lh.pick_up_tips(lh.deck.get_resource("imac_tips")[tips])
                log("Loading nickel columns")
                if samples_in_24well:
                    await sample_mix_24well(CYCLES_SAMPLE, VOL_SAMPLE, tubes, DELAY, FLOWRATE)
                else:
                    await sample_mix_50ml(CYCLES_SAMPLE, VOL_SAMPLE, tubes, DELAY, FLOWRATE)
                await lh.drop_tips(lh.deck.get_resource("imac_tips")[tips])
            await lh.pick_up_tips96(lh.deck.get_resource("imac_tips"))
        elif samples_in_24well_offset:
            log("Loading nickel columns from the 24-well plate with an XY offset")
            await plate_mix(CYCLES_SAMPLE, VOL_SAMPLE, "nickel_sample_plate", DELAY, FLOWRATE, plate_offset=plate_offset)
        else:
            await plate_mix(CYCLES_SAMPLE, VOL_SAMPLE, "nickel_sample_plate", DELAY, FLOWRATE)

        log("Washing nickel columns")
        await plate_mix(CYCLES_WASH, VOL_WASH, "nickel_wash1_plate", DELAY, FLOWRATE, plate_offset=plate_offset)
        log("Washing nickel columns")
        await plate_mix(CYCLES_WASH, VOL_WASH, "nickel_wash2_plate", DELAY, FLOWRATE, plate_offset=plate_offset)
        log("Eluting nickel columns")
        await plate_mix(CYCLES_ELN, VOL_ELN, "nickel_elution_plate", DELAY, FLOWRATE)
        await lh.dispense96(lh.deck.get_resource("nickel_elution_plate"), volume=0, blow_out_air_volume=10)
        await lh.drop_tips96(lh.deck.get_resource("imac_tips"))


In [ ]:
async def hold(seconds: float, message: str) -> None:
    log(message)
    remaining = float(seconds)
    while remaining > 0:
        step = min(30.0, remaining)
        await asyncio.sleep(step)
        remaining -= step
        if remaining > 0:
            log(f"  {remaining:.0f} s remaining")


async def swell_columns(sec_spots, buffer_wells) -> None:
    n = len(buffer_wells)
    liquid = zero_air_liquid_class(0.0)
    z = end_z(buffer_wells[0], buffer_height_mm)
    log(f"SEC steps 4-5. Pick up SEC tips and aspirate {swell_uL:.0f} uL swelling buffer at {column_flow:.1f} uL/s.")
    await lh.pick_up_tips(sec_spots)
    await lh.aspirate(
        buffer_wells,
        vols=repeat(float(swell_uL), n),
        flow_rates=repeat(column_flow, n),
        liquid_height=repeat(buffer_height_mm, n),
        hamilton_liquid_classes=repeat(liquid, n),
        disable_volume_correction=repeat(True, n),
        transport_air_volume=repeat(0.0, n),
        pull_out_distance_transport_air=repeat(0.0, n),
        swap_speed=repeat(10.0, n),
        settling_time=repeat(0.0, n),
        min_z_endpos=z,
    )
    await hold(
        swell_hold_s,
        f"SEC step 6. Holding SEC tips in the swelling buffer for {swell_hold_s:.0f} s. "
        "The tips should still be submerged. Stop the run if they have retracted.",
    )
    if do_buffer_touchoff:
        log("SEC step 7. Touch off in the buffer so a hanging drop wicks off the SEC tip.")
        await lh.dispense(
            buffer_wells,
            vols=repeat(0.0, n),
            flow_rates=repeat(column_flow, n),
            liquid_height=repeat(buffer_height_mm, n),
            hamilton_liquid_classes=repeat(liquid, n),
            disable_volume_correction=repeat(True, n),
            transport_air_volume=repeat(0.0, n),
            pull_out_distance_transport_air=repeat(0.0, n),
            swap_speed=repeat(10.0, n),
            settling_time=repeat(0.0, n),
            minimum_traverse_height_at_beginning_of_a_command=z,
        )
    await lh.drop_tips(sec_spots, allow_nonzero_volume=True)


async def dispense_into_sec_tips(transfer_spots, source_wells, load_wells, volume: float, what: str) -> None:
    n = len(load_wells)
    liquid = zero_air_liquid_class(0.0)
    source_height = sample_height_mm if what == "sample" else buffer_height_mm
    log(f"Pick up 300 uL tips and aspirate {volume:.0f} uL {what}.")
    await lh.pick_up_tips(transfer_spots)
    await lh.aspirate(
        source_wells,
        vols=repeat(float(volume), n),
        flow_rates=repeat(aspirate_flow, n),
        liquid_height=repeat(source_height, n),
        hamilton_liquid_classes=repeat(liquid, n),
        disable_volume_correction=repeat(True, n),
        transport_air_volume=repeat(0.0, n),
        pull_out_distance_transport_air=repeat(0.0, n),
        swap_speed=repeat(10.0, n),
    )
    log(
        f"Dispense {what} into the SEC tips at {transfer_flow:.0f} uL/s, "
        f"{load_depth_mm:.1f} mm below the opening."
    )
    await lh.dispense(
        load_wells,
        vols=repeat(float(volume), n),
        flow_rates=repeat(transfer_flow, n),
        offsets=repeat(Coordinate(0, 0, -load_depth_mm), n),
        hamilton_liquid_classes=repeat(liquid, n),
        disable_volume_correction=repeat(True, n),
        transport_air_volume=repeat(0.0, n),
        pull_out_distance_transport_air=repeat(0.0, n),
        swap_speed=repeat(10.0, n),
    )
    if what == "sample":
        await lh.discard_tips()
    else:
        await lh.drop_tips(transfer_spots, allow_nonzero_volume=True)


async def push_through96(plate_name: str, liquid_uL: float, where: str) -> None:
    # Mount the whole SEC rack with the 96-head plunger fully up, dispense, and return it.
    # full_blowout retracts the shared plunger, mounts the rack, and lifts to traverse height.
    # The following dispense is a normal downward move of liquid_uL + correction.
    plate = lh.deck.get_resource(plate_name)
    sec_rack = lh.deck.get_resource("sec_tips")
    push_uL = float(liquid_uL) + float(correction_uL)
    if push_uL <= 0 or push_uL > 1000:
        raise ValueError(f"Column push into the {where} is {push_uL:.0f} uL. It must be 1-1000 uL.")
    if any(tracker.has_tip for tracker in lh.head.values()):
        raise RuntimeError("The 8 channels still have tips. Return them before the 96 head moves.")
    if any(lh.head96[i].has_tip for i in range(96)):
        raise RuntimeError("The 96 head already has tips. Return them before the column push.")

    flow_liquid = zero_air_liquid_class(0.0)
    touch_liquid = zero_air_liquid_class(touch_air_uL)
    z = end_z(plate["A1"][0], flow_height_mm)
    log(
        "96 head full_blowout: plunger all the way up, then the whole SEC rack is mounted. "
        "Every tip physically in the rack is picked up and will dispense the same volume. "
        "Leave unused positions empty."
    )
    await lh.pick_up_tips96(sec_rack, tip_pickup_method="full_blowout")
    try:
        log(
            f"Dispense {push_uL:.0f} uL into every well of the {where} "
            f"({liquid_uL:.0f} uL liquid + {correction_uL:.0f} uL correction) at {column_flow:.1f} uL/s."
        )
        await lh.dispense96(
            plate,
            volume=push_uL,
            flow_rate=column_flow,
            liquid_height=flow_height_mm,
            hlc=flow_liquid,
            disable_volume_correction=True,
            transport_air_volume=0.0,
            min_z_endpos=z,
            use_lld=False,
            pull_out_distance_transport_air=0.0,
            cut_off_speed=min(float(column_flow), 2.0),
            swap_speed=10.0,
            settling_time=0.1,
        )
        await hold(
            flow_hold_s,
            f"Settling {flow_hold_s:.0f} s after the column flow. Tips should still be in the {where}.",
        )
        if touch_air_uL > 0:
            log(f"Touch off into the {where} with {touch_air_uL:.0f} uL transport air, then wait {touch_hold_s:.0f} s.")
            try:
                await lh.dispense96(
                    plate,
                    volume=0.0,
                    flow_rate=column_flow,
                    liquid_height=flow_height_mm,
                    hlc=touch_liquid,
                    disable_volume_correction=True,
                    transport_air_volume=float(touch_air_uL),
                    min_z_endpos=z,
                    use_lld=False,
                    pull_out_distance_transport_air=2.0,
                    cut_off_speed=min(float(column_flow), 2.0),
                    swap_speed=10.0,
                    settling_time=min(float(touch_hold_s), FIRMWARE_MAX_SETTLE_S),
                )
            except Exception as exc:
                log(f"Touch-off returned: {exc}. Continuing so the SEC tips can be put back.")
            extra_wait = touch_hold_s - FIRMWARE_MAX_SETTLE_S
            if extra_wait > 0:
                await hold(extra_wait, f"Waiting the rest of the {touch_hold_s:.0f} s touch-off.")
    finally:
        log("Return the SEC rack.")
        await lh.return_tips96(allow_nonzero_volume=True)


class TransferTipCursor:
    # Hand out 300 uL transfer-tip columns, using the first rack until it is finished.

    def __init__(self) -> None:
        self.racks = ["sec_transfer_tips"]
        try:
            lh.deck.get_resource("sec_transfer_tips_2")
        except Exception:
            pass
        else:
            self.racks.append("sec_transfer_tips_2")
        self.rack_index = 0
        self.column = 1

    def take(self, rows: list[str]):
        if self.column > 12:
            self.rack_index += 1
            self.column = 1
        if self.rack_index >= len(self.racks):
            raise RuntimeError(
                "Not enough 300 uL transfer tips. Set Transfer tips 2 when the run needs more than one rack."
            )
        names = [f"{row}{self.column}" for row in rows]
        rack = self.racks[self.rack_index]
        log(f"Transfer tips from {rack} column {self.column} ({names[0]}:{names[-1]}).")
        spots = tip_spots(rack, names)
        self.column += 1
        return spots


async def prepare_column(names: list[str], tips: TransferTipCursor) -> dict:
    # Swell one plate column and load sample. The 96-head push happens after every column is loaded.
    rows = [name[0] for name in names]
    sec_spots = tip_spots("sec_tips", names)
    buffer_wells = plate_wells("sec_buffer_plate", names)
    sample_wells = plate_wells("nickel_elution_plate", names)
    load_wells = [LOAD_WELLS[name] for name in names]
    log(f"SEC column {names[0][1:]} wells {names[0]}:{names[-1]}")

    await swell_columns(sec_spots, buffer_wells)

    log("SEC steps 8-11. Transfer nickel eluate into the pierced SEC tips.")
    sample_spots = tips.take(rows)
    buffer_spots = tips.take(rows)
    await dispense_into_sec_tips(sample_spots, sample_wells, load_wells, sample_uL, "sample")
    return {
        "buffer_spots": buffer_spots,
        "buffer_wells": buffer_wells,
        "load_wells": load_wells,
    }


def require_sec_labware() -> None:
    missing = []
    for name in (
        "sec_tips",
        "sec_transfer_tips",
        "sec_buffer_plate",
        "sec_waste_plate",
        "sec_elution_plate",
        "nickel_elution_plate",
    ):
        try:
            lh.deck.get_resource(name)
        except Exception:
            missing.append(name)
    if missing or not LOAD_WELLS:
        detail = ", ".join(missing) if missing else "SEC load guide"
        raise RuntimeError(
            "SEC cleanup is selected, but this deck was set up without SEC labware "
            f"({detail}). Restart the kernel, leave SEC cleanup checked, and click Setup Deck."
        )


async def run_sec() -> None:
    require_sec_labware()
    if any(tracker.has_tip for tracker in lh.head.values()):
        raise RuntimeError("A channel still has a tip. Return or trash it before SEC.")
    if any(lh.head96[i].has_tip for i in range(96)):
        raise RuntimeError("The 96 head still has tips. Return them before SEC.")
    if sample_uL > VOL_ELN:
        raise ValueError(
            f"SEC sample volume ({sample_uL:.0f} uL) is larger than the nickel elution ({VOL_ELN} uL)."
        )

    names = elution_wells()
    void_liquid = max(0.0, float(void_uL) - float(sample_uL)) if do_void_elution else 0.0
    buffer_need = swell_uL + void_liquid + elution_uL + 200
    log("Piercing is skipped. SEC barriers must already be open.")
    log("SEC sample is the nickel elution plate, in the wells that received the nickel elution.")
    log("The column push uses the 96 head on the whole SEC rack. Every column is swelled and loaded before that push.")
    log(
        f"Fill each used buffer well with at least {buffer_need:.0f} uL "
        f"({swell_uL:.0f} swell + {void_liquid:.0f} void elution + {elution_uL:.0f} elution + 200 uL extra)."
    )
    log("SEC wells: " + ", ".join(names))
    tips = TransferTipCursor()
    prepared = []
    for group in column_groups(names):
        prepared.append(await prepare_column(group, tips))

    log("SEC steps 12-17. Flow the sample into the column. Void buffer collects in the waste plate.")
    await push_through96("sec_waste_plate", sample_uL, "waste plate")

    if do_void_elution and void_liquid <= 0:
        log("Sample volume is at least the void volume, so the void elution is skipped.")
    elif void_liquid > 0:
        log(
            f"Void elution. Sample {sample_uL:.0f} uL is below the {void_uL:.0f} uL void, "
            f"so {void_liquid:.0f} uL of buffer is pushed to waste before collection."
        )
        for column in prepared:
            await dispense_into_sec_tips(
                column["buffer_spots"],
                column["buffer_wells"],
                column["load_wells"],
                void_liquid,
                "void buffer",
            )
        await push_through96("sec_waste_plate", void_liquid, "waste plate")

    log("SEC steps 18-26. Add elution buffer and collect the target in the SEC elution plate.")
    for column in prepared:
        await dispense_into_sec_tips(
            column["buffer_spots"],
            column["buffer_wells"],
            column["load_wells"],
            elution_uL,
            "elution buffer",
        )
    await push_through96("sec_elution_plate", elution_uL, "elution plate")
    log("SEC method finished. SEC tips are back in the rack.")


async def teach_load_height() -> None:
    # Park a 300 uL tip just below the SEC opening and leave it there so the height can be checked.
    global teach_tip_spot
    require_sec_labware()
    if any(tracker.has_tip for tracker in lh.head.values()):
        raise RuntimeError("A channel still has a tip. Return or trash it first.")
    names = elution_wells()[:1]
    teach_tip_spot = tip_spots("sec_transfer_tips", names)[0]
    load_well = LOAD_WELLS[names[0]]
    z = end_z(load_well, z_offset=-load_depth_mm)
    liquid = zero_air_liquid_class(0.0)
    log(f"Teaching load height at {names[0]}, {load_depth_mm:.1f} mm below the SEC opening (tip end Z {z:.2f} mm).")
    await lh.pick_up_tips([teach_tip_spot])
    await lh.dispense(
        [load_well],
        vols=[0.0],
        flow_rates=[transfer_flow],
        offsets=[Coordinate(0, 0, -load_depth_mm)],
        hamilton_liquid_classes=[liquid],
        disable_volume_correction=[True],
        transport_air_volume=[0.0],
        pull_out_distance_transport_air=[0.0],
        min_z_endpos=z,
    )
    log("Tip is holding at the load height. Look at the pierced barrier, then click Return teach tip.")


async def return_teach_tip() -> None:
    global teach_tip_spot
    if teach_tip_spot is None:
        raise RuntimeError("No teach tip is tracked. Use Return 8-head tips if a tip is still on a channel.")
    await lh.drop_tips([teach_tip_spot], allow_nonzero_volume=True)
    teach_tip_spot = None
    log("Returned the teach tip.")


async def run_purification(transfer_buffers_enabled: bool, regenerate_columns_enabled: bool):
    if not deck_is_ready or lh is None:
        raise RuntimeError("Deck is not ready. Click Setup Deck first.")
    if RUN_SEC:
        require_sec_labware()

    await transfer_buffers_step(transfer_buffers_enabled)
    await run_nickel()
    if RUN_SEC:
        log("Nickel elution is done. Starting SEC cleanup from the nickel elution plate.")
        await run_sec()
    await regenerate_columns_step(regenerate_columns_enabled)


In [ ]:
def apply_ui_settings(settings: dict) -> None:
    global number_of_samples, samples_in_50ml, samples_in_24well, samples_in_24well_offset
    global wash_in_24well_offset, sample_volume, first_well, RUN_SEC
    global tip_carrier_rail, nickel_plate_carrier_rail, nickel_regen_carrier_rail, sec_plate_carrier_rail
    global tube_carrier_rail_1, tube_carrier_rail_2, tube_carrier_rail_3, tube_carrier_rail_4, trough_carrier_rail
    global imac_tip_pos, sec_tip_pos, buffer_transfer_tip_pos, sec_transfer_tip_pos, sec_transfer_tip_pos_2
    global sec_buffer_plate_pos, sec_waste_plate_pos, sec_elution_plate_pos
    global VOL_EQL, VOL_SAMPLE, VOL_WASH, VOL_ELN
    global CYCLES_EQL, CYCLES_SAMPLE, CYCLES_WASH, CYCLES_ELN, NICKEL_REPEATS, FLOWRATE, DELAY
    global swell_uL, void_uL, sample_uL, elution_uL, correction_uL, do_void_elution, do_buffer_touchoff
    global column_flow, transfer_flow, aspirate_flow
    global swell_hold_s, flow_hold_s, drop_hold_s, touch_hold_s
    global drop_uL, touch_air_uL, airgap_margin_uL
    global load_depth_mm, buffer_height_mm, sample_height_mm, flow_height_mm
    global DEVELOPMENT_MODE

    samples = int(settings["number_of_samples"])
    if samples < 1 or samples > 48:
        raise ValueError("Samples must be between 1 and 48.")

    first = str(settings["first_well"]).strip().upper()
    if len(first) < 2 or first[0] not in "ABCDEFGH" or not first[1:].isdigit():
        raise ValueError("First well must look like A1 to H12.")
    first_col = int(first[1:])
    if first_col < 1 or first_col > 12:
        raise ValueError("First well must be in columns 1 to 12.")

    in_50 = bool(settings["samples_in_50ml"])
    in_24 = bool(settings["samples_in_24well"])
    in_offset = bool(settings["samples_in_24well_offset"])
    if sum([in_50, in_24, in_offset]) > 1:
        raise ValueError("Choose only one of: 50 mL tubes, 24-well plate (8-channel), or 24-well plate offset.")
    if in_24 or in_offset:
        if first[0] not in "ABCD" or not 1 <= first_col <= 6:
            raise ValueError("For 24-well samples, first well must be A1 to D6.")

    run_sec = bool(settings["run_sec"])
    vol_eln = int(settings["vol_eln"])
    sec_sample = float(settings["sample_uL"])
    sec_elution = float(settings["elution_uL"])
    if run_sec:
        if sec_sample > 300 or sec_elution > 350:
            raise ValueError("SEC sample must be <= 300 uL and SEC elution <= 350 uL for the 300 uL transfer tips.")
        if sec_sample > vol_eln:
            raise ValueError(f"SEC sample volume ({sec_sample:.0f} uL) is larger than the nickel elution ({vol_eln} uL).")
        if float(settings["touch_air_uL"]) > 50:
            raise ValueError("Touch-off transport air must be <= 50 uL (Hamilton limit).")
        if float(settings["drop_uL"]) < 20 or float(settings["drop_uL"]) > 30:
            raise ValueError("Hanging-drop aspiration should stay in the DPX range of 20-30 uL.")

    tip_positions = [int(settings["imac_tip_pos"]), int(settings["buffer_transfer_tip_pos"])]
    transfer_2 = int(settings["sec_transfer_tip_pos_2"])
    if run_sec:
        tip_positions.extend([int(settings["sec_tip_pos"]), int(settings["sec_transfer_tip_pos"])])
        if transfer_2:
            tip_positions.append(transfer_2)
    unique_positions("Tip", tip_positions)
    if run_sec:
        unique_positions(
            "SEC plate",
            [
                int(settings["sec_buffer_plate_pos"]),
                int(settings["sec_waste_plate_pos"]),
                int(settings["sec_elution_plate_pos"]),
            ],
        )

    rails = {
        "Tip carrier": int(settings["tip_carrier_rail"]),
        "Nickel plates": int(settings["nickel_plate_carrier_rail"]),
        "Nickel regen": int(settings["nickel_regen_carrier_rail"]),
        "Tube carrier 1": int(settings["tube_carrier_rail_1"]),
        "Trough carrier": int(settings["trough_carrier_rail"]),
    }
    if samples > 12:
        rails["Tube carrier 2"] = int(settings["tube_carrier_rail_2"])
    if samples > 24:
        rails["Tube carrier 3"] = int(settings["tube_carrier_rail_3"])
    if samples > 36:
        rails["Tube carrier 4"] = int(settings["tube_carrier_rail_4"])
    if run_sec:
        rails["SEC plates"] = int(settings["sec_plate_carrier_rail"])
    used = {}
    for label, rail in rails.items():
        if rail in used:
            raise ValueError(f"{label} rail {rail} is already used by {used[rail]}.")
        used[rail] = label

    number_of_samples = samples
    first_well = first
    samples_in_50ml = in_50
    samples_in_24well = in_24
    samples_in_24well_offset = in_offset
    if samples_in_50ml and len(fifty_ml_wells()) != samples:
        raise ValueError(f"With first well {first}, 50 mL loading can reach {len(fifty_ml_wells())} samples.")
    if samples_in_24well:
        loaded = [well for group in groups_24well() for well in group["plate_wells"]]
        if len(loaded) != samples:
            raise ValueError(f"With first well {first}, 24-well loading can reach {len(loaded)} samples.")
    if not samples_in_50ml and not samples_in_24well:
        rows_left = 8 - "ABCDEFGH".index(first[0])
        wells_left = rows_left + (12 - first_col) * 8
        if in_offset:
            rows_left = 4 - "ABCD".index(first[0])
            wells_left = rows_left + (6 - first_col) * 4
        if samples > wells_left:
            raise ValueError(f"With first well {first}, the maximum number of samples is {wells_left}.")

    if run_sec:
        columns_needed = transfer_tip_columns_needed(elution_wells())
        rack_columns = 24 if transfer_2 else 12
        if columns_needed > rack_columns:
            raise ValueError(
                f"This run needs {columns_needed} columns of 300 uL transfer tips. "
                "One rack covers 12 columns. Set Transfer tips 2 to a free slot when more are required."
            )

    wash_in_24well_offset = bool(settings["wash_in_24well_offset"])
    sample_volume = float(settings["sample_volume_ml"])
    RUN_SEC = run_sec
    VOL_EQL = int(settings["vol_eql"])
    VOL_SAMPLE = int(settings["vol_sample"])
    VOL_WASH = int(settings["vol_wash"])
    VOL_ELN = vol_eln
    CYCLES_EQL = int(settings["cycles_eql"])
    CYCLES_SAMPLE = int(settings["cycles_sample"])
    CYCLES_WASH = int(settings["cycles_wash"])
    CYCLES_ELN = int(settings["cycles_eln"])
    NICKEL_REPEATS = int(settings["nickel_repeats"])
    FLOWRATE = float(settings["flowrate"])
    DELAY = float(settings["delay"])
    swell_uL = float(settings["swell_uL"])
    void_uL = float(settings["void_uL"])
    sample_uL = sec_sample
    elution_uL = sec_elution
    correction_uL = float(settings["correction_uL"])
    do_void_elution = bool(settings["do_void_elution"])
    do_buffer_touchoff = bool(settings["do_buffer_touchoff"])
    column_flow = float(settings["column_flow"])
    transfer_flow = float(settings["transfer_flow"])
    aspirate_flow = float(settings["aspirate_flow"])
    swell_hold_s = float(settings["swell_hold_s"])
    flow_hold_s = float(settings["flow_hold_s"])
    drop_hold_s = float(settings["drop_hold_s"])
    touch_hold_s = float(settings["touch_hold_s"])
    drop_uL = float(settings["drop_uL"])
    touch_air_uL = float(settings["touch_air_uL"])
    airgap_margin_uL = float(settings["airgap_margin_uL"])
    load_depth_mm = float(settings["load_depth_mm"])
    buffer_height_mm = float(settings["buffer_height_mm"])
    sample_height_mm = float(settings["sample_height_mm"])
    flow_height_mm = float(settings["flow_height_mm"])
    tip_carrier_rail = int(settings["tip_carrier_rail"])
    nickel_plate_carrier_rail = int(settings["nickel_plate_carrier_rail"])
    nickel_regen_carrier_rail = int(settings["nickel_regen_carrier_rail"])
    sec_plate_carrier_rail = int(settings["sec_plate_carrier_rail"])
    tube_carrier_rail_1 = int(settings["tube_carrier_rail_1"])
    tube_carrier_rail_2 = int(settings["tube_carrier_rail_2"])
    tube_carrier_rail_3 = int(settings["tube_carrier_rail_3"])
    tube_carrier_rail_4 = int(settings["tube_carrier_rail_4"])
    trough_carrier_rail = int(settings["trough_carrier_rail"])
    imac_tip_pos = int(settings["imac_tip_pos"])
    sec_tip_pos = int(settings["sec_tip_pos"])
    buffer_transfer_tip_pos = int(settings["buffer_transfer_tip_pos"])
    sec_transfer_tip_pos = int(settings["sec_transfer_tip_pos"])
    sec_transfer_tip_pos_2 = transfer_2
    sec_buffer_plate_pos = int(settings["sec_buffer_plate_pos"])
    sec_waste_plate_pos = int(settings["sec_waste_plate_pos"])
    sec_elution_plate_pos = int(settings["sec_elution_plate_pos"])
    DEVELOPMENT_MODE = bool(settings["development_mode"])


In [ ]:
STYLE = {"description_width": "170px"}


def flt(value, low, high, step, description):
    return widgets.BoundedFloatText(
        value=value, min=low, max=high, step=step, description=description, style=STYLE
    )


def intbox(value, low, high, description):
    return widgets.BoundedIntText(value=value, min=low, max=high, description=description, style=STYLE)


title = widgets.HTML("<h3>Nickel purification with optional SEC</h3>")
instructions = widgets.HTML(
    "<b>Workflow:</b> set samples and check SEC if this run should continue into size exclusion. "
    "Click <b>Setup Deck</b>. For SEC, teach the load height on an empty tip and return that tip. "
    "Then click <b>Run purification</b>. "
    "Restart the kernel before Setup Deck if you change the sample count, rails, tip slots, or the SEC checkbox. "
    "Pierce SEC barriers before the run. SEC takes sample from the nickel elution plate."
)

samples_field = intbox(number_of_samples, 1, 48, "Samples")
first_well_field = widgets.Text(value=first_well, description="First well", style=STYLE)
run_96well_toggle = widgets.Checkbox(
    value=not (samples_in_50ml or samples_in_24well or samples_in_24well_offset),
    description="Samples in 96-well plate",
    layout={"width": "360px"},
)
samples_in_50ml_toggle = widgets.Checkbox(
    value=samples_in_50ml, description="Samples in 50 mL tubes", layout={"width": "360px"}
)
samples_in_24well_toggle = widgets.Checkbox(
    value=samples_in_24well,
    description="Samples in 24-well plate (8-channel)",
    layout={"width": "420px"},
)
samples_in_24well_offset_toggle = widgets.Checkbox(
    value=samples_in_24well_offset,
    description="Samples in 24-well plate, XY offset",
    layout={"width": "420px"},
)
wash_in_24well_offset_toggle = widgets.Checkbox(
    value=wash_in_24well_offset,
    description="Wash with 24-well XY offset",
    layout={"width": "420px"},
)
sec_toggle = widgets.Checkbox(
    value=RUN_SEC,
    description="SEC cleanup after nickel",
    layout={"width": "360px"},
)
transfer_buffers_toggle = widgets.Checkbox(value=False, description="Transfer buffers before nickel")
regenerate_columns_toggle = widgets.Checkbox(value=True, description="Regenerate nickel columns at end")
development_toggle = widgets.Checkbox(value=False, description="Development mode (verbose logs)")


def _on_run_type_changed(change):
    toggles = [
        run_96well_toggle,
        samples_in_50ml_toggle,
        samples_in_24well_toggle,
        samples_in_24well_offset_toggle,
    ]
    if change["new"]:
        for toggle in toggles:
            if toggle is not change["owner"]:
                toggle.value = False
        return
    if not any(toggle.value for toggle in toggles):
        run_96well_toggle.value = True


for _toggle in (
    run_96well_toggle,
    samples_in_50ml_toggle,
    samples_in_24well_toggle,
    samples_in_24well_offset_toggle,
):
    _toggle.observe(_on_run_type_changed, names="value")

flowrate_field = flt(FLOWRATE, 0.3, 1000, 0.1, "Nickel flow uL/s")
delay_field = flt(DELAY, 0, 120, 0.5, "Nickel delay s")
sample_volume_ml_field = flt(sample_volume, 0.1, 50, 0.1, "Sample vol mL")
vol_eql_field = intbox(VOL_EQL, 1, 2000, "VOL_EQL")
vol_sample_field = intbox(VOL_SAMPLE, 1, 2000, "VOL_SAMPLE")
vol_wash_field = intbox(VOL_WASH, 1, 2000, "VOL_WASH")
vol_eln_field = intbox(VOL_ELN, 1, 2000, "VOL_ELN")
cycles_eql_field = intbox(CYCLES_EQL, 1, 20, "Equil cycles")
cycles_sample_field = intbox(CYCLES_SAMPLE, 1, 30, "Sample cycles")
cycles_wash_field = intbox(CYCLES_WASH, 1, 20, "Wash cycles")
cycles_eln_field = intbox(CYCLES_ELN, 1, 20, "Elution cycles")
nickel_repeats_field = intbox(NICKEL_REPEATS, 1, 20, "Nickel repeats")

column_field = widgets.Dropdown(
    options=list(SEC_COLUMNS.keys()),
    value=DEFAULT_COLUMN,
    description="SEC column",
    style=STYLE,
    layout={"width": "560px"},
)
swell_field = flt(swell_uL, 50, 900, 1, "Swell uL")
void_field = flt(void_uL, 50, 400, 1, "Void uL")
sec_sample_field = flt(sample_uL, 1, 300, 1, "SEC sample uL")
sec_elution_field = flt(elution_uL, 1, 350, 1, "SEC elution uL")
correction_field = flt(correction_uL, 0, 200, 1, "Correction uL")
void_toggle = widgets.Checkbox(value=do_void_elution, description="Void elution when sample < void")
touch_toggle = widgets.Checkbox(value=do_buffer_touchoff, description="Touch off in buffer after swell")
column_flow_field = flt(column_flow, 1, 50, 0.5, "Column uL/s")
transfer_flow_field = flt(transfer_flow, 20, 250, 5, "Into-tip uL/s")
aspirate_flow_field = flt(aspirate_flow, 10, 400, 5, "Aspirate uL/s")
swell_hold_field = flt(swell_hold_s, 0, 600, 1, "Swell hold s")
flow_hold_field = flt(flow_hold_s, 0, 120, 1, "Flow hold s")
drop_hold_field = flt(drop_hold_s, 0, 30, 0.5, "Drop hold s")
touch_hold_field = flt(touch_hold_s, 0, 30, 0.5, "Touch hold s")
drop_field = flt(drop_uL, 20, 30, 1, "Drop uL")
touch_air_field = flt(touch_air_uL, 0, 50, 1, "Touch air uL")
margin_field = flt(airgap_margin_uL, 0, 200, 5, "Air-gap extra")
load_depth_field = flt(load_depth_mm, -10, 40, 0.5, "Load depth mm")
buffer_height_field = flt(buffer_height_mm, 1, 30, 0.5, "Buffer height mm")
sample_height_field = flt(sample_height_mm, 0.5, 20, 0.5, "Sample height mm")
flow_height_field = flt(flow_height_mm, 1, 40, 0.5, "Flow height mm")

tip_rail_field = intbox(tip_carrier_rail, -20, 60, "Tip rail")
nickel_plate_rail_field = intbox(nickel_plate_carrier_rail, -20, 60, "Nickel plate rail")
nickel_regen_rail_field = intbox(nickel_regen_carrier_rail, -20, 60, "Nickel regen rail")
sec_plate_rail_field = intbox(sec_plate_carrier_rail, -20, 60, "SEC plate rail")
tube_rail_1_field = intbox(tube_carrier_rail_1, -20, 60, "Tube 1 rail")
tube_rail_2_field = intbox(tube_carrier_rail_2, -20, 60, "Tube 2 rail")
tube_rail_3_field = intbox(tube_carrier_rail_3, -20, 60, "Tube 3 rail")
tube_rail_4_field = intbox(tube_carrier_rail_4, -20, 60, "Tube 4 rail")
trough_rail_field = intbox(trough_carrier_rail, -20, 60, "Trough rail")
imac_tip_pos_field = intbox(imac_tip_pos, 1, 5, "IMAC tips slot")
sec_tip_pos_field = intbox(sec_tip_pos, 1, 5, "SEC tips slot")
buffer_tip_pos_field = intbox(buffer_transfer_tip_pos, 1, 5, "Buffer tips slot")
sec_transfer_pos_field = intbox(sec_transfer_tip_pos, 1, 5, "SEC transfer slot")
sec_transfer_pos_2_field = intbox(sec_transfer_tip_pos_2, 0, 5, "SEC transfer 2")
sec_buffer_pos_field = intbox(sec_buffer_plate_pos, 1, 5, "Swell plate slot")
sec_waste_pos_field = intbox(sec_waste_plate_pos, 1, 5, "Waste plate slot")
sec_elution_pos_field = intbox(sec_elution_plate_pos, 1, 5, "SEC elution slot")


def _on_column_changed(change):
    preset = SEC_COLUMNS[change["new"]]
    swell_field.value = preset["swell_uL"]
    void_field.value = preset["void_uL"]


column_field.observe(_on_column_changed, names="value")

setup_button = widgets.Button(description="Setup Deck", button_style="info")
teach_button = widgets.Button(description="Teach load height", button_style="warning", disabled=True)
return_teach_button = widgets.Button(description="Return teach tip", button_style="warning", disabled=True)
run_button = widgets.Button(description="Run purification", button_style="success", disabled=True)
return_96_button = widgets.Button(description="Return 96-head tips", button_style="warning", disabled=True)
trash_96_button = widgets.Button(description="Trash 96-head tips", button_style="danger", disabled=True)
return_8_button = widgets.Button(description="Return 8-head tips", button_style="warning", disabled=True)
trash_8_button = widgets.Button(description="Trash 8-head tips", button_style="danger", disabled=True)

status_output = widgets.Output(
    layout={"border": "1px solid #ddd", "padding": "8px", "max_height": "360px", "overflow": "auto"}
)
deck_summary = widgets.Textarea(
    value="",
    description="Deck",
    layout={"width": "100%", "height": "220px"},
    disabled=True,
)
run_state_label = widgets.HTML("")


def _collect_settings():
    return {
        "number_of_samples": samples_field.value,
        "first_well": first_well_field.value,
        "samples_in_50ml": samples_in_50ml_toggle.value,
        "samples_in_24well": samples_in_24well_toggle.value,
        "samples_in_24well_offset": samples_in_24well_offset_toggle.value,
        "wash_in_24well_offset": wash_in_24well_offset_toggle.value,
        "run_sec": sec_toggle.value,
        "sample_volume_ml": sample_volume_ml_field.value,
        "vol_eql": vol_eql_field.value,
        "vol_sample": vol_sample_field.value,
        "vol_wash": vol_wash_field.value,
        "vol_eln": vol_eln_field.value,
        "cycles_eql": cycles_eql_field.value,
        "cycles_sample": cycles_sample_field.value,
        "cycles_wash": cycles_wash_field.value,
        "cycles_eln": cycles_eln_field.value,
        "nickel_repeats": nickel_repeats_field.value,
        "flowrate": flowrate_field.value,
        "delay": delay_field.value,
        "swell_uL": swell_field.value,
        "void_uL": void_field.value,
        "sample_uL": sec_sample_field.value,
        "elution_uL": sec_elution_field.value,
        "correction_uL": correction_field.value,
        "do_void_elution": void_toggle.value,
        "do_buffer_touchoff": touch_toggle.value,
        "column_flow": column_flow_field.value,
        "transfer_flow": transfer_flow_field.value,
        "aspirate_flow": aspirate_flow_field.value,
        "swell_hold_s": swell_hold_field.value,
        "flow_hold_s": flow_hold_field.value,
        "drop_hold_s": drop_hold_field.value,
        "touch_hold_s": touch_hold_field.value,
        "drop_uL": drop_field.value,
        "touch_air_uL": touch_air_field.value,
        "airgap_margin_uL": margin_field.value,
        "load_depth_mm": load_depth_field.value,
        "buffer_height_mm": buffer_height_field.value,
        "sample_height_mm": sample_height_field.value,
        "flow_height_mm": flow_height_field.value,
        "tip_carrier_rail": tip_rail_field.value,
        "nickel_plate_carrier_rail": nickel_plate_rail_field.value,
        "nickel_regen_carrier_rail": nickel_regen_rail_field.value,
        "sec_plate_carrier_rail": sec_plate_rail_field.value,
        "tube_carrier_rail_1": tube_rail_1_field.value,
        "tube_carrier_rail_2": tube_rail_2_field.value,
        "tube_carrier_rail_3": tube_rail_3_field.value,
        "tube_carrier_rail_4": tube_rail_4_field.value,
        "trough_carrier_rail": trough_rail_field.value,
        "imac_tip_pos": imac_tip_pos_field.value,
        "sec_tip_pos": sec_tip_pos_field.value,
        "buffer_transfer_tip_pos": buffer_tip_pos_field.value,
        "sec_transfer_tip_pos": sec_transfer_pos_field.value,
        "sec_transfer_tip_pos_2": sec_transfer_pos_2_field.value,
        "sec_buffer_plate_pos": sec_buffer_pos_field.value,
        "sec_waste_plate_pos": sec_waste_pos_field.value,
        "sec_elution_plate_pos": sec_elution_pos_field.value,
        "development_mode": development_toggle.value,
    }


def _set_busy(busy: bool) -> None:
    setup_button.disabled = busy
    ready = deck_is_ready and not busy
    teach_button.disabled = not ready
    return_teach_button.disabled = not ready
    run_button.disabled = not ready
    return_96_button.disabled = not ready
    trash_96_button.disabled = not ready
    return_8_button.disabled = not ready
    trash_8_button.disabled = not ready


def _show_error(exc: Exception) -> None:
    status_output.append_stdout(f"Failed: {exc}\n")
    if DEVELOPMENT_MODE:
        status_output.append_stdout("\n" + "".join(traceback.format_exc()) + "\n")


async def _setup_async():
    _set_busy(True)
    if not DEVELOPMENT_MODE:
        status_output.clear_output()
    status_output.append_stdout("Setting up deck...\n")
    try:
        apply_ui_settings(_collect_settings())
        result = await setup_deck()
        if isinstance(result, dict):
            status_output.append_stdout(result["status"] + "\n")
            deck_summary.value = result["deck"] or "Deck is ready."
        else:
            status_output.append_stdout(str(result) + "\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _run_async():
    _set_busy(True)
    if not DEVELOPMENT_MODE:
        status_output.clear_output()
    status_output.append_stdout("Running purification...\n")
    status_output.append_stdout(time.ctime() + "\n")
    try:
        apply_ui_settings(_collect_settings())
        settings = _collect_settings()
        estimate_s = estimate_run_seconds(
            settings,
            transfer_buffers_toggle.value,
            regenerate_columns_toggle.value,
        )
        finish = time.strftime("%I:%M %p", time.localtime(time.time() + estimate_s)).lstrip("0")
        duration = format_duration(estimate_s)
        run_state_label.value = f"<b>Status:</b> Running. Estimated finish {finish} ({duration})."
        status_output.append_stdout(f"Estimated time to completion: {duration} (about {finish}).\n")
        if settings["samples_in_50ml"]:
            run_type = "50 mL tubes"
        elif settings["samples_in_24well"]:
            run_type = "24-well plate (8-channel)"
        elif settings["samples_in_24well_offset"]:
            run_type = "24-well plate (XY offset)"
        else:
            run_type = "96-well plate"
        lines = [
            "Run summary:",
            f" - Sample loading: {run_type}",
            f" - Samples: {settings['number_of_samples']} starting at {str(settings['first_well']).strip()}",
            f" - Sample volume recorded: {settings['sample_volume_ml']} mL",
            f" - Transfer buffers: {'yes' if transfer_buffers_toggle.value else 'no'}",
            f" - Regenerate nickel columns: {'yes' if regenerate_columns_toggle.value else 'no'}",
            f" - SEC cleanup: {'yes' if settings['run_sec'] else 'no'}",
        ]
        if settings["run_sec"]:
            void_liquid = (
                max(0.0, float(settings["void_uL"]) - float(settings["sample_uL"]))
                if settings["do_void_elution"]
                else 0.0
            )
            lines.extend(
                [
                    f" - SEC column preset: {column_field.value}",
                    f" - SEC swell {settings['swell_uL']:.0f} uL, sample {settings['sample_uL']:.0f} uL, elution {settings['elution_uL']:.0f} uL",
                    f" - Void {settings['void_uL']:.0f} uL, void elution liquid {void_liquid:.0f} uL",
                    f" - Correction added to each SEC dispense: {settings['correction_uL']:.0f} uL",
                    f" - SEC sample wells are on the nickel elution plate",
                    " - Piercing tool steps: skipped",
                ]
            )
        status_output.append_stdout("\n".join(lines) + "\n\n")
        await run_purification(
            transfer_buffers_enabled=transfer_buffers_toggle.value,
            regenerate_columns_enabled=regenerate_columns_toggle.value,
        )
        status_output.append_stdout("Run complete.\n")
        status_output.append_stdout(time.ctime() + "\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        run_state_label.value = "<b>Status:</b> Idle"
        _set_busy(False)


async def _teach_async():
    _set_busy(True)
    try:
        apply_ui_settings(_collect_settings())
        await teach_load_height()
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _return_teach_async():
    _set_busy(True)
    try:
        await return_teach_tip()
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _return_96_async():
    _set_busy(True)
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup Deck first.")
        await lh.return_tips96(allow_nonzero_volume=True)
        status_output.append_stdout("Returned 96-head tips.\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _trash_96_async():
    _set_busy(True)
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup Deck first.")
        await lh.discard_tips96()
        status_output.append_stdout("Trashed 96-head tips.\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _return_8_async():
    _set_busy(True)
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup Deck first.")
        await lh.return_tips(allow_nonzero_volume=True)
        status_output.append_stdout("Returned 8-head tips.\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _trash_8_async():
    _set_busy(True)
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup Deck first.")
        await lh.discard_tips(allow_nonzero_volume=True)
        status_output.append_stdout("Trashed 8-head tips.\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


eta_label = widgets.HTML("")


def _eta_html(seconds: float) -> str:
    return (
        "<div style='font-size:18px;padding:10px 14px;margin:4px 0 8px;"
        "background:#eef5ff;border:1px solid #b7d0ef;border-radius:6px;'>"
        f"<b>Estimated time to completion: {format_duration(seconds)}</b>"
        "<div style='font-size:13px;color:#333;margin-top:4px;'>"
        "Liquid flow, programmed waits, and about 8 s of arm travel per move."
        "</div></div>"
    )


def _refresh_eta(_change=None) -> None:
    try:
        seconds = estimate_run_seconds(
            _collect_settings(),
            transfer_buffers_toggle.value,
            regenerate_columns_toggle.value,
        )
    except Exception:
        eta_label.value = (
            "<div style='font-size:18px;padding:10px 14px;margin:4px 0 8px;"
            "background:#fff6e8;border:1px solid #e6c98a;border-radius:6px;'>"
            "<b>Estimated time to completion:</b> check the sample settings"
            "</div>"
        )
        return
    eta_label.value = _eta_html(seconds)


_eta_widgets = [
    samples_field,
    first_well_field,
    run_96well_toggle,
    samples_in_50ml_toggle,
    samples_in_24well_toggle,
    samples_in_24well_offset_toggle,
    sec_toggle,
    transfer_buffers_toggle,
    regenerate_columns_toggle,
    flowrate_field,
    delay_field,
    vol_eql_field,
    vol_sample_field,
    vol_wash_field,
    vol_eln_field,
    cycles_eql_field,
    cycles_sample_field,
    cycles_wash_field,
    cycles_eln_field,
    nickel_repeats_field,
    swell_field,
    void_field,
    sec_sample_field,
    sec_elution_field,
    correction_field,
    void_toggle,
    touch_toggle,
    column_flow_field,
    transfer_flow_field,
    aspirate_flow_field,
    swell_hold_field,
    flow_hold_field,
    touch_hold_field,
    touch_air_field,
]
for _eta_widget in _eta_widgets:
    _eta_widget.observe(_refresh_eta, names="value")
_refresh_eta()

setup_button.on_click(lambda _: asyncio.create_task(_setup_async()))
teach_button.on_click(lambda _: asyncio.create_task(_teach_async()))
return_teach_button.on_click(lambda _: asyncio.create_task(_return_teach_async()))
run_button.on_click(lambda _: asyncio.create_task(_run_async()))
return_96_button.on_click(lambda _: asyncio.create_task(_return_96_async()))
trash_96_button.on_click(lambda _: asyncio.create_task(_trash_96_async()))
return_8_button.on_click(lambda _: asyncio.create_task(_return_8_async()))
trash_8_button.on_click(lambda _: asyncio.create_task(_trash_8_async()))

nickel_advanced = widgets.Accordion(
    children=[
        widgets.VBox([sample_volume_ml_field, flowrate_field, delay_field]),
        widgets.VBox([vol_eql_field, vol_sample_field, vol_wash_field, vol_eln_field]),
        widgets.VBox([
            cycles_eql_field,
            cycles_sample_field,
            cycles_wash_field,
            cycles_eln_field,
            nickel_repeats_field,
        ]),
    ]
)
nickel_advanced.set_title(0, "Nickel process")
nickel_advanced.set_title(1, "Nickel volumes")
nickel_advanced.set_title(2, "Nickel cycles")

sec_advanced = widgets.Accordion(
    children=[
        widgets.VBox([
            column_flow_field,
            transfer_flow_field,
            aspirate_flow_field,
            swell_hold_field,
            flow_hold_field,
            drop_hold_field,
            touch_hold_field,
            drop_field,
            touch_air_field,
            margin_field,
        ]),
        widgets.VBox([
            load_depth_field,
            buffer_height_field,
            sample_height_field,
            flow_height_field,
        ]),
    ]
)
sec_advanced.set_title(0, "SEC flows and waits")
sec_advanced.set_title(1, "SEC heights")

deck_advanced = widgets.Accordion(
    children=[
        widgets.VBox([
            tip_rail_field,
            nickel_plate_rail_field,
            nickel_regen_rail_field,
            sec_plate_rail_field,
            tube_rail_1_field,
            tube_rail_2_field,
            tube_rail_3_field,
            tube_rail_4_field,
            trough_rail_field,
            widgets.HTML("<b>Tip carrier slots</b> (1 = front, 5 = back). Transfer 2 = 0 leaves that rack off."),
            imac_tip_pos_field,
            sec_tip_pos_field,
            buffer_tip_pos_field,
            sec_transfer_pos_field,
            sec_transfer_pos_2_field,
            widgets.HTML("<b>SEC plate slots</b> on the SEC carrier. Sample is the nickel elution plate."),
            sec_buffer_pos_field,
            sec_waste_pos_field,
            sec_elution_pos_field,
        ])
    ]
)
deck_advanced.set_title(0, "Rails and slots")

controls = widgets.VBox(
    [
        title,
        instructions,
        widgets.HTML("<b>Samples</b>"),
        samples_field,
        first_well_field,
        run_96well_toggle,
        samples_in_50ml_toggle,
        samples_in_24well_toggle,
        samples_in_24well_offset_toggle,
        wash_in_24well_offset_toggle,
        transfer_buffers_toggle,
        regenerate_columns_toggle,
        development_toggle,
        nickel_advanced,
        widgets.HTML("<b>SEC cleanup</b>"),
        sec_toggle,
        column_field,
        swell_field,
        void_field,
        sec_sample_field,
        sec_elution_field,
        correction_field,
        void_toggle,
        touch_toggle,
        widgets.HTML(
            "Correction is extra microlitres the SEC tip dispenses beyond the liquid volume. "
            "DPX says this is usually needed to finish pushing liquid through the bed. Start at 0 and increase it on a dye test."
        ),
        sec_advanced,
        deck_advanced,
        eta_label,
        widgets.HBox([setup_button, teach_button, return_teach_button, run_button]),
        widgets.HTML(
            "<b>Tip recovery.</b> Trash 96 discards whatever the 96-head is holding, including SEC columns if those tips are mounted. Use Return 96 to put a rack back."
        ),
        widgets.HBox([return_96_button, trash_96_button, return_8_button, trash_8_button]),
        deck_summary,
        run_state_label,
        status_output,
    ]
)
display(controls)
